In [ ]:
%load_ext autoreload
%autoreload 2
import dt4dds_benchmark
import plotly.express as px
import pandas as pd

data = dt4dds_benchmark.analysis.Dataset.combine(*[dt4dds_benchmark.pipelines.HDF5Manager(f'./data/{w}/{s}.hdf5').get_data() for s in (
    'modulation_medium', 'ldpc_medium', 'dbgps_low', 'dbgps_medium', 'dbgps_high',
) for w in (
    'basic',
    'cdhit',
    'clover',
    'lsh',
    'starcode',
)])

In [ ]:
data.combined_results

### get the threshold values by codec, clustering, and scenario

In [ ]:
df = data.get_fits_by_group(['codec.type', 'codec.name', 'clustering.name', 'clustering.type'], on='workflow.overall_rate', additional_agg={'code_rate': 'mean'})
df['code_rate'] = df['code_rate'].map('{:.2f}'.format)

df

In [ ]:
idf = df[['codec.type', 'code_rate', 'clustering.name', 'clustering.type', 'threshold']].copy()

# convert to wide by codec name and type
sdf = idf.pivot_table(columns=['clustering.type', 'clustering.name'], index=['codec.type', 'code_rate'], values='threshold', aggfunc='first').reset_index()
sdf

### plot only best-performing clustering and the basic clustering

In [ ]:
plotdf = df.loc[df['clustering.type'] != 'MMseqs2'].copy()
plotdf['clustergroup'] = plotdf['clustering.type']
idf = df.copy()
idf['clustergroup'] = idf['clustering.type']

# keep only rows where the threshold is highest per codec.type and codec.name, but always also include the BasicSet row
plotdf = plotdf.loc[plotdf.groupby(['codec.type', 'codec.name'])['threshold'].idxmax()]
plotdf['clustergroup'] = 'optimal'
plotdf = pd.concat([plotdf, idf.loc[df['clustering.type'] == 'BasicSet']])
plotdf['codec.type'] = plotdf['codec.type'].str.replace('Goldman', 'GM').replace('YinYang', 'YY')
plotdf = plotdf.sort_values(['codec.type', 'code_rate', 'clustering.type'])

fig = dt4dds_benchmark.analysis.plotting.tiered_bar(
    plotdf,
    "codec.type",
    "code_rate",
    "threshold",
    color_by = "clustergroup",
    color_discrete_map={'BasicSet': '#636363', 'optimal': '#31a354'},
)
fig.update_yaxes(
    range=[0, 0.15],
    title='Error rate per nt',
    # type="log",
)
fig.update_layout(
    width=320,
    height=140,
    margin=dict(l=0, r=1, t=10, b=30),
    showlegend=False,
)

fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)
fig.update_xaxes(
    tickfont_size=28/3, 
    tickangle=0,
)
fig.show()
fig.write_image('./figures/best.svg')

# export data
plotdf.to_csv('./figures/individual_plot.csv', index=False)